# Tasneem – Llama-3-8B-Instruct Evaluation
Evaluating medical QA performance on open-ended and MCQ questions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────
OUT_DIR      = "/content/drive/MyDrive/Colab Notebooks/Activity_1_UFS_1_2026/Tasneem"
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/Activity_1_UFS_1_2026/questions_w_answers.jsonl"
MCQ_NO_ANS   = "/content/drive/MyDrive/Colab Notebooks/Activity_1_UFS_1_2026/Moaath_Questions_163_189_without_Correct_Answer.csv"
MCQ_ANS      = "/content/drive/MyDrive/Colab Notebooks/Activity_1_UFS_1_2026/Moaath_Questions_163_189_with_Correct_Answer.csv"

MODEL_TAG    = "llama3"

In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import json, os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ── Load open questions (lines 100-116) ────────────────────────────────
open_qs = []
with open(DATASET_PATH) as fh:
    for idx, raw in enumerate(fh):
        if 117 <= idx <= 133:
            obj = json.loads(raw)
            open_qs.append({"id": idx + 1,
                            "question": obj["Question"],
                            "must_have": obj["Must_have"]})

print(f"Loaded {len(open_qs)} open questions")

In [ ]:
# ── Load MCQ data ───────────────────────────────────────────────────────
mcq_q = pd.read_csv(MCQ_NO_ANS)
mcq_a = pd.read_csv(MCQ_ANS)
print(f"MCQ rows: {len(mcq_q)}")

In [ ]:
# ── Helpers ─────────────────────────────────────────────────────────────
def pick_letter(text: str):
    """Return first A-F letter found in model output."""
    text = text.upper()
    for ch in "ABCDEF":
        for pat in (f"{ch})", f"{ch}.", f"ANSWER: {ch}", f"OPTION {ch}", f" {ch} "):
            if pat in text:
                return ch
    return None


def must_have_score(response: str, keywords: list) -> float:
    """Fraction of must-have keywords present in response."""
    if not keywords:
        return 0.0
    resp_low = response.lower()
    hits = sum(
        1 for kw in keywords
        if sum(1 for w in kw.lower().split() if w in resp_low)
           >= max(1, len(kw.split()) // 2)
    )
    return hits / len(keywords)

In [ ]:
# ── Load model ──────────────────────────────────────────────────────────
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_cfg, device_map="auto"
)
print("Model ready")

In [ ]:
# ── Open-ended evaluation ────────────────────────────────────────────────
open_records = []

for q in open_qs:
    messages = [
        {"role": "system", "content": "You are a knowledgeable medical assistant."},
        {"role": "user",   "content": q["question"]}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, temperature=0.3, do_sample=True)

    reply = tokenizer.decode(out[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    score = must_have_score(reply, q["must_have"])

    open_records.append({"id": q["id"], "question": q["question"],
                         "answer": reply, "must_have_score": score})

open_df = pd.DataFrame(open_records)
open_df.to_csv(f"{OUT_DIR}/llama3_open_tasneem.csv", index=False)
print("Open eval done – mean score:", open_df.must_have_score.mean().round(4))

In [ ]:
# ── MCQ evaluation ───────────────────────────────────────────────────────
mcq_records = []

for i, row in mcq_q.iterrows():
    prompt_text = (
        f"Choose ONE letter (A-F) only.\n\n{row['Question_text']}\n"
        f"A) {row['(A)']}\nB) {row['(B)']}\nC) {row['(C)']}\n"
        f"D) {row['(D)']}\nE) {row['(E)']}\nF) {row['(F)']}\n\nAnswer:"
    )
    messages = [
        {"role": "system", "content": "You are a medical exam assistant. Reply with a single letter."},
        {"role": "user",   "content": prompt_text}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=5, do_sample=False)

    reply = tokenizer.decode(out[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip().upper()
    pred  = pick_letter(reply) or (reply[-1] if reply else None)
    gold  = mcq_a.loc[i, "Correct_answer"].strip().upper()

    mcq_records.append({"question": row["Question_text"],
                        f"{MODEL_TAG}_prediction": pred,
                        "correct": gold,
                        "score": pred == gold})

mcq_df = pd.DataFrame(mcq_records)
mcq_df.to_csv(f"{OUT_DIR}/llama3_mcq_tasneem.csv", index=False)
print("MCQ accuracy:", mcq_df.score.mean().round(4))

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────
summary = pd.DataFrame([{
    "model":              MODEL_TAG,
    "mcq_accuracy":       mcq_df.score.mean(),
    "mcq_correct":        int(mcq_df.score.sum()),
    "mcq_total":          len(mcq_df),
    "open_mean":          open_df.must_have_score.mean(),
    "open_median":        open_df.must_have_score.median(),
    "open_std":           open_df.must_have_score.std(),
    "open_min":           open_df.must_have_score.min(),
    "open_max":           open_df.must_have_score.max(),
    "open_good_0.5":      int((open_df.must_have_score >= 0.5).sum()),
    "open_excellent_0.8": int((open_df.must_have_score >= 0.8).sum()),
}])

summary.to_csv(f"{OUT_DIR}/llama3_summary.csv", index=False)
print(summary.T)

In [ ]:
# ── Cleanup GPU memory ───────────────────────────────────────────────────
import gc
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print("GPU cleared")